# 02.3 髓系细胞亚群精细分析（Myeloid Subclustering）

本 notebook 在 `02` 的 broad 注释基础上，专门针对髓系细胞进行亚群再聚类、marker 辅助判读与亚型命名。

运行顺序建议：`01 -> 02 -> 02.3`

主要输入：`results/intermediate/integration/annotated_broad.h5ad`
主要输出：
- `results/intermediate/integration/subset_myeloid_02_3.h5ad`
- `results/tables/integration/myeloid_02_3_cluster_markers.tsv`
- `results/tables/integration/myeloid_02_3_cluster_annotation_template.tsv`
- `results/figures/integration/myeloid_02_3/*.png|*.pdf`


## Section 0: 环境准备与全局配置


In [ ]:
# === Section 0: 导入依赖与路径配置 ===
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from IPython.display import display

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor='white')

class Config:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
    INTEGRATION_DIR = PROJECT_ROOT / 'results' / 'intermediate' / 'integration'
    TABLE_DIR = PROJECT_ROOT / 'results' / 'tables' / 'integration'
    FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'integration' / 'myeloid_02_3'
    CHECKPOINT = INTEGRATION_DIR / 'annotated_broad.h5ad'
    OUT_H5AD = INTEGRATION_DIR / 'subset_myeloid_02_3.h5ad'
    OUT_MARKER = TABLE_DIR / 'myeloid_02_3_cluster_markers.tsv'
    OUT_CLUSTER_SUMMARY = TABLE_DIR / 'myeloid_02_3_cluster_summary.tsv'
    OUT_ANNOT_TEMPLATE = TABLE_DIR / 'myeloid_02_3_cluster_annotation_template.tsv'
    OUT_UMAP = FIG_DIR / 'myeloid_02_3_umap'
    OUT_DOTPLOT = FIG_DIR / 'myeloid_02_3_top_markers_dotplot'
    OUT_MARKER_PANEL = FIG_DIR / 'myeloid_02_3_marker_panel'
    OUT_PATIENT_CONTRIB = FIG_DIR / 'myeloid_02_3_patient_contribution'
    OUT_SUBTYPE_UMAP = FIG_DIR / 'myeloid_02_3_subtype_umap'
    OUT_SUBTYPE_DOTPLOT = FIG_DIR / 'myeloid_02_3_subtype_dotplot'

Config.TABLE_DIR.mkdir(parents=True, exist_ok=True)
Config.FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(base_path: Path):
    base_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(base_path.with_suffix('.png'), bbox_inches='tight')
    plt.savefig(base_path.with_suffix('.pdf'), bbox_inches='tight')
    plt.close("all")

print(f'Project root: {Config.PROJECT_ROOT}')
print(f'Input checkpoint: {Config.CHECKPOINT}')


## Section 0.5: 读取 checkpoint 并校验输入


In [ ]:
# === Section 0.5: load annotated_broad checkpoint ===
if not Config.CHECKPOINT.exists():
    raise FileNotFoundError('Missing annotated_broad.h5ad. Please run notebook 02 first.')

adata = sc.read_h5ad(Config.CHECKPOINT)
required_obs = ['broad_celltype', 'sample_id', 'patient_id']
missing_cols = [c for c in required_obs if c not in adata.obs.columns]
if missing_cols:
    raise ValueError(f'Missing required obs columns: {missing_cols}')

myeloid_mask = adata.obs['broad_celltype'].astype(str).str.lower().eq('myeloid')
mye = adata[myeloid_mask].copy()
if mye.n_obs == 0:
    raise ValueError("No myeloid cells found in adata.obs['broad_celltype']")

print(f'Total cells in annotated_broad: {adata.n_obs}')
print(f'Myeloid cells selected: {mye.n_obs}')


## Section 1: 髓系亚群重聚类

标准流程：归一化 -> 对数变换 -> HVG -> PCA -> neighbors -> UMAP -> Leiden。


In [ ]:
# === Section 1: myeloid re-clustering ===
sc.pp.normalize_total(mye, target_sum=1e4)
sc.pp.log1p(mye)

if 'sample_id' in mye.obs.columns and mye.obs['sample_id'].nunique() > 1:
    sc.pp.highly_variable_genes(mye, n_top_genes=3000, flavor='seurat', batch_key='sample_id')
else:
    sc.pp.highly_variable_genes(mye, n_top_genes=3000, flavor='seurat')

mye = mye[:, mye.var['highly_variable']].copy()
sc.pp.scale(mye, max_value=10)
sc.tl.pca(mye, svd_solver="arpack")
sc.pp.neighbors(mye, n_neighbors=15, n_pcs=30)
sc.tl.umap(mye, min_dist=0.35)
sc.tl.leiden(mye, resolution=0.6, key_added='myeloid_leiden')

cluster_counts = mye.obs['myeloid_leiden'].value_counts().sort_index()
print(cluster_counts)
cluster_counts.rename('n_cells').to_csv(Config.OUT_CLUSTER_SUMMARY, sep='	', header=True)
print(f'Saved cluster summary: {Config.OUT_CLUSTER_SUMMARY}')


## Section 2: marker 差异分析与基础判读


In [ ]:
# === Section 2: differential expression ===
sc.tl.rank_genes_groups(mye, groupby='myeloid_leiden', method='wilcoxon', key_added='rank_genes_myeloid_02_3')
marker_df = sc.get.rank_genes_groups_df(mye, group=None, key='rank_genes_myeloid_02_3')
marker_df.to_csv(Config.OUT_MARKER, sep='	', index=False)
print(f'Saved marker table: {Config.OUT_MARKER}')
display(marker_df.head(10))


## Section 2B: 预置髓系 marker 面板与患者贡献图（人工判读）

风格对齐 `02.2`：先给 marker panel 图谱，再看 cluster 的患者构成。


In [ ]:
# === Section 2B: marker panel + patient contribution ===
MYELOID_MARKERS_02_3 = {
    'Pan_Myeloid': ['LYZ', 'TYROBP', 'AIF1'],
    'Mono_Inflammatory': ['FCN1', 'VCAN', 'S100A8', 'S100A9', 'IL1B'],
    'Neutrophils': ['FCGR3B', 'CSF3R', 'CXCL8'],
    'C1QC_TAM': ['C1QA', 'C1QB', 'APOE', 'TREM2', 'CD163', 'MRC1'],
    'SPP1_TAM': ['SPP1', 'PLAUR', 'CTSD'],
    'cDC2': ['CD1C', 'CLEC10A', 'FCER1A', 'HLA-DRA', 'CD74'],
    'pDC': ['LILRA4', 'CLEC4C', 'IRF7', 'GZMB'],
    'Mast_cells': ['TPSAB1', 'TPSB2', 'CPA3', 'KIT'],
}

valid = {}
for k, genes in MYELOID_MARKERS_02_3.items():
    present = [g for g in genes if g in mye.var_names]
    if present:
        valid[k] = present
    else:
        print(f'[Marker missing] {k}: {genes}')

sc.pl.dotplot(mye, valid, groupby='myeloid_leiden', standard_scale='var', color_map='Reds', show=False)
save_figure(Config.OUT_MARKER_PANEL)
print(f'Saved marker panel: {Config.OUT_MARKER_PANEL}.png/.pdf')

if 'patient_id' in mye.obs.columns:
    patient_tab = pd.crosstab(mye.obs['myeloid_leiden'], mye.obs['patient_id'])
    patient_prop = patient_tab.div(patient_tab.sum(axis=1), axis=0)
    ax = patient_prop.plot(kind='bar', stacked=True, figsize=(9, 4))
    ax.set_title('Patient contribution by myeloid_leiden')
    ax.set_ylabel('Proportion')
    plt.legend(title='patient_id', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_figure(Config.OUT_PATIENT_CONTRIB)
    print(f'Saved patient contribution: {Config.OUT_PATIENT_CONTRIB}.png/.pdf')


## Section 2C: 半自动 subtype 命名模板（先建议后人工微调）

本节会先生成 cluster->subtype 建议表，再保留手动修改入口，风格与 `02.2` 的“手动填写映射”一致。


In [ ]:
# === Section 2C: semi-auto annotation template ===
SIGNATURES = {
    'Neutrophils/MDSC': ['S100A8', 'S100A9', 'FCGR3B', 'CSF3R', 'CXCL8'],
    'FCN1+ Monocytes': ['FCN1', 'VCAN', 'IL1B', 'EREG', 'AREG'],
    'C1QC+ TAMs': ['C1QA', 'C1QB', 'C1QC', 'APOE', 'TREM2', 'CD163'],
    'SPP1+ TAMs': ['SPP1', 'PLAUR', 'CTSD'],
    'cDC2': ['CD1C', 'CLEC10A', 'FCER1A', 'HLA-DRA', 'CD74'],
    'pDC': ['LILRA4', 'CLEC4C', 'IRF7', 'GZMB'],
    'Mast cells': ['TPSAB1', 'TPSB2', 'CPA3', 'KIT'],
}

score_cols = []
for subtype, genes in SIGNATURES.items():
    present = [g for g in genes if g in mye.var_names]
    if len(present) < 2:
        print(f'[Skip score] {subtype}: too few genes present -> {present}')
        continue
    col = f"score__{subtype.replace(' ', '_').replace('+', 'pos').replace('/', '_')}"
    sc.tl.score_genes(mye, gene_list=present, score_name=col, use_raw=False)
    score_cols.append((subtype, col))

cluster_rows = []
clusters = sorted(
    mye.obs['myeloid_leiden'].astype(str).unique(),
    key=lambda x: int(x) if x.isdigit() else x,
)

for cl in clusters:
    sub = mye[mye.obs['myeloid_leiden'].astype(str) == cl]
    row = {'cluster': cl, 'n_cells': int(sub.n_obs)}

    best_name = 'unassigned'
    best_score = -np.inf

    for subtype, col in score_cols:
        s = float(np.nanmean(sub.obs[col]))
        row[f'mean_{subtype}'] = s
        if np.isfinite(s) and s > best_score:
            best_score = s
            best_name = subtype

    row['suggested_subtype'] = best_name
    cluster_rows.append(row)

cluster_suggestion_df = pd.DataFrame(cluster_rows).sort_values(by='cluster')
cluster_suggestion_df.to_csv(Config.OUT_ANNOT_TEMPLATE, sep='	', index=False)
print(f'Saved annotation template: {Config.OUT_ANNOT_TEMPLATE}')

if {'cluster', 'n_cells', 'suggested_subtype'}.issubset(cluster_suggestion_df.columns):
    display(cluster_suggestion_df[['cluster', 'n_cells', 'suggested_subtype']])
else:
    display(cluster_suggestion_df)

if not score_cols:
    print('[Warning] No signature scores computed. All clusters default to unassigned. Please review marker panel manually.')

# ---- manual adjustment entry (same spirit as 02.2 Section 6C) ----
cluster_annotation = {str(r.cluster): r.suggested_subtype for r in cluster_suggestion_df.itertuples()}
# Example: cluster_annotation["3"] = "Neutrophils/MDSC"
# Example: del cluster_annotation["7"]  # if you want to exclude a cluster

mye.obs['myeloid_cell_subtype'] = mye.obs['myeloid_leiden'].astype(str).map(cluster_annotation)
mye.obs['myeloid_cell_subtype'] = mye.obs['myeloid_cell_subtype'].fillna('unassigned')
print('Current subtype counts:')
print(mye.obs['myeloid_cell_subtype'].value_counts())


## Section 3: 可视化与结果导出

同时展示 `myeloid_leiden` 和 `myeloid_cell_subtype`，方便核对半自动命名合理性。


In [ ]:
# === Section 3: visualization export ===
# Build plot keys defensively to avoid KeyError when some columns are absent.
candidate_keys = ['myeloid_leiden', 'sample_id', 'patient_id', 'myeloid_cell_subtype', 'pni_level']
plot_keys = [k for k in candidate_keys if k in mye.obs.columns]

if not plot_keys:
    raise ValueError('No valid obs columns found for UMAP plotting in Section 3.')

sc.pl.umap(mye, color=plot_keys, wspace=0.45, show=False)
save_figure(Config.OUT_SUBTYPE_UMAP)
print(f'Saved UMAP: {Config.OUT_SUBTYPE_UMAP}.png/.pdf')

# Subtype-level marker dotplot requires myeloid_cell_subtype.
if 'myeloid_cell_subtype' in mye.obs.columns:
    subtype_series = mye.obs['myeloid_cell_subtype'].astype(str)
    subtype_order = [x for x in subtype_series.value_counts().index.tolist() if x != 'unassigned']

    if subtype_order:
        # Keep a stable order for plotting while preserving unassigned at tail.
        categories = subtype_order + (['unassigned'] if 'unassigned' in subtype_series.values else [])
        mye.obs['myeloid_cell_subtype'] = pd.Categorical(subtype_series, categories=categories, ordered=True)

        FINAL_MARKERS = {
            'Neutrophils_MDSC': ['S100A8', 'S100A9', 'FCGR3B', 'CSF3R'],
            'FCN1_Monocytes': ['FCN1', 'VCAN', 'IL1B', 'EREG'],
            'C1QC_TAMs': ['C1QA', 'C1QB', 'C1QC', 'APOE', 'TREM2'],
            'DC_pDC': ['CD1C', 'FCER1A', 'LILRA4', 'CLEC4C'],
            'Mast_cells': ['TPSAB1', 'TPSB2', 'CPA3', 'KIT'],
        }
        valid_final = {k: [g for g in v if g in mye.var_names] for k, v in FINAL_MARKERS.items()}
        valid_final = {k: v for k, v in valid_final.items() if len(v) > 0}

        if valid_final:
            sc.pl.dotplot(
                mye,
                valid_final,
                groupby='myeloid_cell_subtype',
                standard_scale='var',
                color_map='Reds',
                show=False,
            )
            save_figure(Config.OUT_SUBTYPE_DOTPLOT)
            print(f'Saved subtype dotplot: {Config.OUT_SUBTYPE_DOTPLOT}.png/.pdf')
        else:
            print('[Skip] No valid genes found for FINAL_MARKERS in mye.var_names.')
    else:
        print('[Skip] No assigned subtype categories available for subtype dotplot.')
else:
    print('[Skip] myeloid_cell_subtype not found. Run Section 2C before subtype dotplot.')

# rank_genes_groups dotplot requires Section 2 key.
if 'rank_genes_myeloid_02_3' in mye.uns:
    sc.pl.rank_genes_groups_dotplot(mye, n_genes=5, key='rank_genes_myeloid_02_3', show=False)
    save_figure(Config.OUT_DOTPLOT)
    print(f'Saved rank-genes dotplot: {Config.OUT_DOTPLOT}.png/.pdf')
else:
    print('[Skip] rank_genes_myeloid_02_3 not found in mye.uns. Run Section 2 first.')


## Section 4: 保存髓系子集 checkpoint


In [ ]:
# === Section 4: save checkpoint ===
if 'myeloid_cell_subtype' not in mye.obs.columns:
    mye.obs['myeloid_cell_subtype'] = 'unassigned_' + mye.obs['myeloid_leiden'].astype(str)

mye.write_h5ad(Config.OUT_H5AD)
print(f'Saved subset object: {Config.OUT_H5AD}')
print('Done: 02.3 myeloid subclustering completed.')
